# Exploratory Data Analysis — sample_submission.csv dataset

**Goal:** Explore the Song Lyrics dataset to understand its structure, content, and quality before building a model to classify songs by genre (`tag`).

## Dataset overview
- **Source:** Genius Song Lyrics dataset (`data/train.csv`, `data/test.csv`)
- **Target:** `tag` — genre label (`pop`, `rock`, `rap`, `rb`, `country`, `misc`)
- **Features:**
  | Column | Description |
  |---|---|
  | `id` | Unique song identifier |
  | `title` | Song title |
  | `artist` | Performing artist |
  | `year` | Release year |
  | `views` | Genius page views (popularity proxy) |
  | `features` | Featured/collaborating artists |
  | `lyrics` | Full song lyrics (raw text) |
  | `tag` | Genre label (target variable) |

## Questions this notebook aims to answer
1. **Class balance** — how are genres distributed? Is `tag` imbalanced?
2. **Text characteristics** — how does lyric length, vocabulary size, and repetitiveness vary by genre?
3. **Missing/dirty data** — are there empty lyrics, duplicates, non-English text, or malformed entries?
4. **Temporal trends** — how does genre popularity or lyrical style shift across `year`?
5. **Popularity signal** — does `views` correlate with genre, lyric length, or era?
6. **Vocabulary & language** — what words/n-grams are most distinctive per genre (candidates for feature engineering)?
7. **Train vs. test consistency** - o train.csv and test.csv (implied by sample_submission.csv) share similar distributions of year, views, and lyric length?
8. **Artist/genre leakage** - does the same artist appear under multiple tag values, or does the same artist show up in both train and test?
9. **Genius formatting artifacts** - lyrics scraped from Genius typically contain bracketed annotations ([Chorus], [Verse 1], [Produced by...]) 
10. **Duplicate/near-duplicate lyrics** - covers, remixes, or scraping errors can produce near-identical lyrics under different ids
11. **Structural/repetition patterns** - chorus repetition rate, line count, average line length — these often differ meaningfully by genre 
12. **Explicit/profanity rate by genre** - often strong signal for genre classification

## Approach
We will start with structural checks (shape, dtypes, nulls, duplicates), then move to text-specific EDA (length distributions, word frequencies, TF-IDF-style genre signatures), then try to answer those specific questions and close with visualizations that motivate the modeling choices for genre classification.


# <font color='#BFD72F' size=5>1. Data Understanding</font> <a class="anchor" id="P1"></a>
  
[Back to TOC](#toc)

In [1]:
import pandas as pd 

df_train = pd.read_csv('/Users/cicerosantos/Documents/GitHub/Text-Mining-/data/train.csv')
df_test = pd.read_csv('/Users/cicerosantos/Documents/GitHub/Text-Mining-/data/test.csv')

### <font color='#BFD72F' size=5>1.1 General Exploration</font> <a class="anchor" id="P11"></a>

In [2]:
df_train.info()
df_train.describe()

<class 'pandas.DataFrame'>
RangeIndex: 134967 entries, 0 to 134966
Data columns (total 8 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   title     134965 non-null  str  
 1   tag       134967 non-null  str  
 2   artist    134967 non-null  str  
 3   year      134967 non-null  int64
 4   views     134967 non-null  int64
 5   features  134967 non-null  str  
 6   lyrics    134967 non-null  str  
 7   id        134967 non-null  int64
dtypes: int64(3), str(5)
memory usage: 8.2 MB


,year,views,id
count,134967.000000,1.349670e+05,1.349670e+05
mean,2009.467537,3.413344e+03,3.593871e+06
std,46.287743,4.192449e+04,2.323217e+06
min,1.000000,0.000000e+00,2.800000e+01
25%,2008.000000,2.200000e+01,1.414202e+06
50%,2015.000000,8.900000e+01,3.547995e+06
75%,2019.000000,4.730000e+02,5.600458e+06
max,2024.000000,3.604497e+06,7.882826e+06


In [3]:
df_test.info()
df_test.describe()

<class 'pandas.DataFrame'>
RangeIndex: 33742 entries, 0 to 33741
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   title     33742 non-null  str  
 1   artist    33742 non-null  str  
 2   year      33742 non-null  int64
 3   views     33742 non-null  int64
 4   features  33742 non-null  str  
 5   lyrics    33742 non-null  str  
 6   id        33742 non-null  int64
dtypes: int64(3), str(4)
memory usage: 1.8 MB


,year,views,id
count,33742.000000,3.374200e+04,3.374200e+04
mean,2009.649606,3.712460e+03,3.604763e+06
std,40.044269,5.384800e+04,2.323315e+06
min,1.000000,0.000000e+00,7.000000e+00
25%,2008.000000,2.200000e+01,1.421414e+06
50%,2015.000000,8.800000e+01,3.564637e+06
75%,2019.000000,4.600000e+02,5.604923e+06
max,2023.000000,6.877777e+06,7.882715e+06


In [5]:
print(df_train.isnull().sum())
print(df_train.isnull().sum())

title       2
tag         0
artist      0
year        0
views       0
features    0
lyrics      0
id          0
dtype: int64
title       2
tag         0
artist      0
year        0
views       0
features    0
lyrics      0
id          0
dtype: int64


In [11]:
#Analying different types of music genres
print(df_train['tag'].value_counts())

print(df_train['artist'].value_counts())


tag
pop        55742
rap        38584
rock       25332
rb          6203
misc        5640
country     3466
Name: count, dtype: int64
artist
Genius English Translations    555
The Grateful Dead               86
Emily Dickinson                 75
Abraham Lincoln                 59
OCTOBERSFULLMOON                58
                              ... 
Necrolust                        1
The Research                     1
Cerebral Turbulency              1
Transmatic                       1
TheGodlyRiskTakers               1
Name: count, Length: 74916, dtype: int64


Muitos poemas, náo sei se é suposto ser assim ou até se os podemos aproveitar

In [ ]:
df_train['lyrics_length'] = df_train['lyrics'].str.len()

top30_longest = df_train.nlargest(30, 'lyrics_length')[['title', 'artist', 'tag', 'lyrics_length']]
top30_longest

In [ ]:
from langdetect import detect, DetectorFactory, LangDetectException
DetectorFactory.seed = 0  # makes detection deterministic

def detect_lang(text):
    try:
        return detect(text[:300])  # first 300 chars is enough signal and much faster
    except LangDetectException:
        return 'unknown'

df_train['lyrics_lang'] = df_train['lyrics'].apply(detect_lang)
df_train['lyrics_lang'].value_counts()